# Phase 4b - Scaled self-play long-run (the real GO/NO-GO)

What the toy probes settled (see `rl_research/PROBE_ARCHALUDON_2026-06-30.md`):
- **Stability: solved.** The orbit-wars anti-collapse recipe (LR 1e-4, batch 256, 1 epoch,
  entropy floor 0.04) trained `medium` (15.8M) to completion with healthy entropy - no collapse.
- **The in-loop eval was lying.** `quick_eval` silently degraded `kaggle:`/`mirror` opponents
  to the trivial 'first' baseline, so the "92-95%" curves were meaningless. **Now fixed** -
  `quick_eval` runs the real `kaggle:<name>` agent on its own deck (or raises). Re-`git pull`.
- **The real numbers (scripts/eval.py, n=160 vs held-out `kaggle:archaludon`):** small 25.6%,
  medium 28.7% - i.e. both **lose** to the strong matched rule agent, and **capacity barely moved
  it** (25.6 -> 28.7, CIs overlap). The bottleneck is **steps, not params**: we trained ~3M
  decisions; Gerar hit top-2% at 412M (~130x), the big runs at 10-15B. We are badly undertrained.

**So this run is the actual go/no-go: scale the step budget ~6-8x on the stable recipe and read
the REAL `kaggle:archaludon` slope** (now that the in-loop eval is honest). Deck-controlled as
before: Archaludon trainee on the rule agent's exact 60-card list, `kaggle:archaludon` held OUT
of the training pool and used only as the eval yardstick.

**Read `eval: ... kaggle:archaludon`** every 25 iters (this is now the true number):
- climbing through ~40-50% and still rising -> **GO**, fan out A+B at full budget.
- flat in the 25-35% band it started at -> steps alone won't close it -> **BC warm-start**
  (see `orbit-wars-selfplay-lessons` + the BC memory note) is the next lever.

Note: no `--resume` yet, so treat this as one long session (~12-16h at ~40s/iter). Checkpoints
persist to Drive; if it dies, restart loses optimizer/iter progress. Bump `ITERS` to fit budget.

In [ ]:
# Mount Drive (artifact persistence), then clone (or update) the repo and cd in.
import os

from google.colab import drive
drive.mount('/content/drive')

# Everything the runs produce is written under here so it survives the session and
# can be pulled back with scripts/download_artifacts.py. Keep this name in sync with
# DRIVE_BASE in that script ('ptcg_outputs').
DRIVE_OUTPUT = '/content/drive/MyDrive/ptcg_outputs'
for sub in ['', '/logs', '/checkpoints_sp_small', '/ablation_sp']:
    os.makedirs(DRIVE_OUTPUT + sub, exist_ok=True)

REPO_URL = "https://github.com/oshbocker/pokemon_tcg_ai_battle.git"
REPO_DIR = "/content/pokemon_tcg_ai_battle"
if not os.path.isdir(REPO_DIR):
    !git clone --depth 1 {REPO_URL} {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull --ff-only
%cd {REPO_DIR}

assert os.path.isfile("agent/cg/libcg.so"), (
    "agent/cg/libcg.so missing - push it to the repo (see .gitignore exception)."
)
import torch
print("repo:", os.getcwd(), "| Drive:", DRIVE_OUTPUT)
print("torch", torch.__version__, "| CUDA:", torch.cuda.is_available())


In [ ]:
# Hardware topology - record this alongside the results.
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || echo "(no GPU runtime selected)"
!nproc
!lscpu | grep -E "^CPU\(s\):|Thread\(s\) per core|Core\(s\) per socket|Model name"
!free -h | head -2


## Scaled long-run (`medium`, stable recipe, honest in-loop eval)

Same recipe that trained `medium_v2` cleanly, scaled on **iters** (the identified lever).
`--eval-opponents random,heuristic,kaggle:archaludon` now actually runs the hand-coded
Archaludon (the `quick_eval` fix); `mirror` is dropped (it now raises - use the gate signal).
Watch `ent` stays > ~0.05 and the `kaggle:archaludon` eval slope.

**Gate = `--gate-vs pool`** (2026-06-30): `best.pt` is now crowned by the current net's
**weighted win-rate across the whole training league** (self + past checkpoints + the manifest
agents: Starmie/Dragapult/romanrozen/heuristic/random), not just a mirror vs the frozen best.
Watch the per-opponent breakdown in the `gate pool NN% [self .. | starmie .. | ..]` line - it
is a much better proxy for ladder strength than the mirror number, which the old gate could
maximise while staying blind to the aggro decks we meet on the ladder. The gate is **relative**
(promote when the pool score beats the last-promoted checkpoint's), with `--gate-threshold` as a
light absolute floor. `kaggle:archaludon` stays held OUT of the manifest, so the eval column is
still an honest yardstick the gate never optimises against.

In [ ]:
# -- Scaled self-play long-run: the real go/no-go --
ITERS = 1200   # ~6x medium_v2's 200; bump/cut to fit the session budget (~40s/iter)
PROBE_OUT = f'{DRIVE_OUTPUT}/probe_archaludon_medium_long'
LOG = '/content/colab_probe_archaludon_medium_long.txt'
# PFSP (conservative first pass): after a 200-iter static warmup, re-weight the manifest
# opponents by the learner's live win-rate. `var` mode focuses games on contested (~50%)
# matchups. Modest bounds keep it gentle and never starve a pool agent: each opponent's
# weight stays within [0.5x, 2x] its manifest base (--pfsp-wmin 0.5 = the floor). self/past
# are untouched. Watch the new `| mix:` log line (realized share) + any `FORFEIT=` alarm.
!python scripts/train_selfplay.py \
    --deck agent/kaggle_agents/archaludon_deck.csv \
    --opp-manifest agent/opponents/mixed_pool_heldout_archaludon.json \
    --size medium --collector dist --workers 48 \
    --opponent self --league --pool-size 5 --snapshot-every 25 \
    --iters {ITERS} --games-per-iter 256 --epochs 1 --minibatch 256 \
    --lr 1e-4 --lr-final 1e-5 --ent-coef 0.04 --target-entropy 0.15 --target-kl 1.5 \
    --gate --gate-vs pool --gate-every 25 --gate-games 120 --gate-past 2 --gate-threshold 0.4 \
    --eval-every 25 --eval-games 120 \
    --eval-opponents random,heuristic,kaggle:archaludon \
    --pfsp --pfsp-mode var --pfsp-after 200 --pfsp-wmin 0.5 --pfsp-wmax 2.0 --pfsp-ema 0.8 \
    --device cuda --seed 0 \
    --out {PROBE_OUT} 2>&1 | tee {LOG}
!cp {LOG} {DRIVE_OUTPUT}/logs/   # persist log to Drive


## Results are on Drive — pull them to the laptop

Everything the runs produced lives under `MyDrive/ptcg_outputs/` (checkpoints, eval
CSVs, and the `logs/colab_*.txt` records). Nothing needs to be git-pushed from here.
The cell below just confirms what landed; back on the laptop, fetch it with rclone:

```bash
uv run python scripts/download_artifacts.py --logs                 # colab_*.txt -> rl_research/
uv run python scripts/download_artifacts.py --run ablation_sp      # A/B ckpts + CSVs -> outputs/
uv run python scripts/download_artifacts.py --run checkpoints_sp_small
```

Then paste the A/B table + RECOMMENDATION into `ABLATION_OPTION_RANK.md` and commit
the logs. (One-time rclone `gdrive` remote setup is in CLAUDE.md.)

In [ ]:
# Confirm what persisted to Drive.
!ls -lhR {DRIVE_OUTPUT} | sed -n '1,80p'
